<a href="https://colab.research.google.com/github/hsb0205/AI-CLASS/blob/main/WEEK15/imdb_review_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout

In [2]:
# ------------------------------------------------------------
# 1. IMDB 데이터 다운로드
# ------------------------------------------------------------

# keras.datasets.imdb는 케라스에서 제공하는 영화 리뷰 데이터셋이다.
imdb = keras.datasets.imdb

# num_words=10000은 자주 등장하는 상위 10,000개 단어만 사용하겠다는 뜻이다.
# 너무 드물게 등장하는 단어는 제외한다.
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# ------------------------------------------------------------
# 2. 데이터 개수 확인
# ------------------------------------------------------------

# 훈련 데이터와 테스트 데이터의 개수를 출력한다.
print("훈련 데이터 개수, 테스트 데이터 개수")
print(len(x_train), len(x_test))

훈련 데이터 개수, 테스트 데이터 개수
25000 25000


In [4]:
# ------------------------------------------------------------
# 3. 첫 번째 리뷰 데이터 확인
# ------------------------------------------------------------

# x_train[0]은 첫 번째 영화 리뷰이다.
# 실제 문장이 아니라 단어가 정수 번호로 바뀐 상태이다.
print("\n첫 번째 훈련 리뷰")
print(x_train[0])


첫 번째 훈련 리뷰
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [5]:
# ------------------------------------------------------------
# 4. 리뷰 길이 확인
# ------------------------------------------------------------

# 영화 리뷰마다 단어 개수가 다르기 때문에 길이도 다르다.
print("\n첫 번째 리뷰와 두 번째 리뷰의 길이")
print(len(x_train[0]), len(x_train[1]))


첫 번째 리뷰와 두 번째 리뷰의 길이
218 189


In [6]:
# ------------------------------------------------------------
# 5. 라벨 확인
# ------------------------------------------------------------

# y_train에는 정답 라벨이 저장되어 있다.
# 1이면 긍정적인 리뷰, 0이면 부정적인 리뷰이다.
print("\n첫 번째 리뷰와 두 번째 리뷰의 라벨")
print(y_train[0], y_train[1])


첫 번째 리뷰와 두 번째 리뷰의 라벨
1 0


In [7]:
# ------------------------------------------------------------
# 6. 긍정/부정 리뷰 개수 확인
# ------------------------------------------------------------

# np.unique()를 사용하여 라벨 종류와 각 라벨의 개수를 확인한다.
print("\n라벨별 데이터 개수")
print(np.unique(y_train, return_counts=True))


라벨별 데이터 개수
(array([0, 1]), array([12500, 12500]))


In [8]:
# ------------------------------------------------------------
# 7. 정수 인덱스를 단어로 복원하기 위한 딕셔너리 생성
# ------------------------------------------------------------

# word_to_index는 단어와 정수 인덱스가 저장된 딕셔너리이다.
# 예: {"the": 1, "movie": 17, ...}
word_to_index = imdb.get_word_index()

# IMDB 데이터셋에서는 0, 1, 2, 3번 인덱스를 특수한 용도로 사용한다.
# 그래서 기존 단어 인덱스에 3을 더해준다.
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}

# 특수 토큰을 직접 추가한다.
word_to_index["<PAD>"] = 0       # 문장 길이를 맞추기 위해 채우는 값
word_to_index["<START>"] = 1     # 문장의 시작 표시
word_to_index["<UNK>"] = 2       # 모르는 단어
word_to_index["<UNUSED>"] = 3    # 사용하지 않는 토큰

# 단어 → 정수 형태를 정수 → 단어 형태로 바꾼다.
index_to_word = dict((value, key) for (key, value) in word_to_index.items())

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
# ------------------------------------------------------------
# 8. 첫 번째 리뷰를 문장으로 복원
# ------------------------------------------------------------

# x_train[0]에 들어 있는 정수 인덱스를 다시 단어로 바꾼다.
print("\n첫 번째 리뷰 복원 결과")
print(" ".join([index_to_word.get(index, "?") for index in x_train[0]]))


첫 번째 리뷰 복원 결과
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be 

In [10]:
# ------------------------------------------------------------
# 9. 패딩 처리
# ------------------------------------------------------------

# 신경망에 입력하려면 모든 리뷰의 길이가 같아야 한다.
# max_len=100은 모든 리뷰를 단어 100개 길이로 맞춘다는 뜻이다.
max_len = 100

# 길이가 짧으면 0으로 채우고, 길이가 길면 잘라낸다.
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# 패딩 후에는 모든 리뷰 길이가 100으로 같아진다.
print("\n패딩 후 첫 번째 리뷰와 두 번째 리뷰의 길이")
print(len(x_train[0]), len(x_train[1]))

print("\n패딩된 첫 번째 리뷰")
print(x_train[0])


패딩 후 첫 번째 리뷰와 두 번째 리뷰의 길이
100 100

패딩된 첫 번째 리뷰
[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


In [11]:
# ------------------------------------------------------------
# 10. 신경망 모델 만들기
# ------------------------------------------------------------

# vocab_size는 사용할 단어 사전의 크기이다.
vocab_size = 10000

# Sequential은 층을 순서대로 쌓는 모델이다.
model = Sequential()

# Embedding 층
# 정수로 된 단어 인덱스를 64차원의 실수 벡터로 바꾼다.
model.add(Embedding(vocab_size, 64, input_length=max_len))

# Flatten 층
# Embedding 결과를 1차원 형태로 펼친다.
model.add(Flatten())

# Dense 은닉층
# relu 활성화 함수를 사용한다.
model.add(Dense(64, activation="relu"))

# Dropout 층
# 과적합을 줄이기 위해 일부 뉴런을 무작위로 꺼준다.
model.add(Dropout(0.5))

# 출력층
# 이진 분류이므로 출력 노드는 1개이다.
# sigmoid는 0과 1 사이의 값을 출력한다.
model.add(Dense(1, activation="sigmoid"))

# 모델 구조를 출력한다.
print("\n모델 구조")
model.summary()


모델 구조


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
# ------------------------------------------------------------
# 11. 모델 컴파일
# ------------------------------------------------------------

# binary_crossentropy:
# 정답이 0 또는 1인 이진 분류 문제에서 사용하는 손실 함수이다.
#
# optimizer="adam":
# 가중치를 자동으로 업데이트하는 최적화 알고리즘이다.
#
# metrics=["accuracy"]:
# 학습 중 정확도를 함께 확인한다.
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [13]:
# ------------------------------------------------------------
# 12. 모델 학습
# ------------------------------------------------------------

# batch_size=64:
# 데이터를 64개씩 묶어서 학습한다.
#
# epochs=20:
# 전체 훈련 데이터를 20번 반복 학습한다.
#
# validation_data:
# 학습 중 테스트 데이터로 검증 정확도를 확인한다.
history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=20,
    verbose=1,
    validation_data=(x_test, y_test)
)

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.7650 - loss: 0.4639 - val_accuracy: 0.8474 - val_loss: 0.3397
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.9359 - loss: 0.1774 - val_accuracy: 0.8343 - val_loss: 0.4051
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9912 - loss: 0.0315 - val_accuracy: 0.8344 - val_loss: 0.5461
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.9988 - loss: 0.0073 - val_accuracy: 0.8349 - val_loss: 0.6194
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.9999 - loss: 0.0019 - val_accuracy: 0.8382 - val_loss: 0.6602
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.8394 - val_loss: 0.6974
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 1.0000 - loss: 5.5158e-04 - val_accuracy: 0.8394 - val_loss: 0.7323
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9998 - loss: 0.0010 -

In [14]:
# ------------------------------------------------------------
# 13. 모델 평가
# ------------------------------------------------------------

# evaluate()는 손실값과 정확도를 반환한다.
results = model.evaluate(x_test, y_test, verbose=2)

print("\n테스트 평가 결과 [loss, accuracy]")
print(results)

782/782 - 2s - 3ms/step - accuracy: 0.8392 - loss: 1.0530

테스트 평가 결과 [loss, accuracy]
[1.0529881715774536, 0.8391600251197815]


In [15]:
# ------------------------------------------------------------
# 14. 직접 작성한 리뷰로 테스트
# ------------------------------------------------------------

review = """
What can I say about this movie that was already said? It is my favorite time
travel sci-fi, adventure epic comedy in the 80's and I love this movie to death!
When I saw this movie I was thrown out by its theme. An excellent sci-fi,
adventure epic, I LOVE the 80s. It's simple the greatest time travel movie ever
happened in the history of world cinema. I love this movie to death, I love,
LOVE, love it!
"""

# 알파벳, 숫자, 공백만 남기고 특수문자는 제거한다.
# lower()는 모든 문자를 소문자로 바꾼다.
review = re.sub("[^0-9a-zA-Z ]", "", review).lower()

# 리뷰 문장을 정수 인덱스로 변환할 리스트
review_encoding = []

# 리뷰의 단어를 하나씩 꺼내서 정수 인덱스로 바꾼다.
for w in review.split():

    # 단어가 딕셔너리에 없으면 <UNK>에 해당하는 2를 사용한다.
    index = word_to_index.get(w, 2)

    # 10000번 이하의 단어만 사용한다.
    if index <= vocab_size:
        review_encoding.append(index)
    else:
        review_encoding.append(word_to_index["<UNK>"])


# 모델 입력은 2차원 형태이어야 하므로 리스트로 한 번 더 감싼다.
test_input = pad_sequences([review_encoding], maxlen=max_len)

# predict()는 긍정일 확률을 출력한다.
value = model.predict(test_input)

print("\n직접 작성한 리뷰 예측값")
print(value)

# 예측값이 0.5보다 크면 긍정, 아니면 부정으로 판단한다.
if value > 0.5:
    print("긍정적인 리뷰입니다.")
else:
    print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step

직접 작성한 리뷰 예측값
[[1.]]
긍정적인 리뷰입니다.
